# Info Extraction
- Fetch key info from API
- Get Financial Statements and data related to the past 30 days
- Get Quarterly Data

## Verify
- Statements are released on the same day
- Determine analysis period for stock_hist_info()

## Improvement
- FMP contains stats from anaylist maybe we could also use those

In [5]:
from pathlib import Path
import pandas as pd
import requests

In [6]:
class API:

    KEY = "wUvM2M29ZVxHDvK8IRp2P7iyrT8uhQG4"
    BASE_URL = "https://financialmodelingprep.com/stable/"
    
    # Might need to track number of API CALLS PER MINUTE
    
    def fetch(endpoint="", query="",verbose=False):
        url = f"{API.BASE_URL}{endpoint}?{query}&apikey={API.KEY}"
        if verbose:
            print(f"\trequest url:{url}")
        response = requests.get(url)
        json_response = response.json() 
        return pd.DataFrame(json_response)
           
    def fetch_sp500_constituent_info():
        df_constituent_full_info = API.fetch("sp500-constituent")
        df_constituent_full_info.set_index('symbol', inplace=True)
        df_constituent_light_info = df_constituent_full_info[['sector','subSector']] 
        return df_constituent_light_info

    # The "date" in the responses of fetch_ticker_key_metrics() and fetch_ticker_financial_ratios() represents the end date of the period the financial statement covers 
    def fetch_ticker_key_metrics(symbol,period="quarter"):
        endpoint = "key-metrics"
        query = f"symbol={symbol}&period={period}"
        df_key_metrics = API.fetch(endpoint,query)
        return API.set_index_and_sort(df_key_metrics)

    def fetch_ticker_financial_ratios(symbol,period="quarter"):
        endpoint = "ratios"
        query = f"symbol={symbol}&period={period}"
        df_financial_ratio = API.fetch(endpoint,query)
        return API.set_index_and_sort(df_financial_ratio)

    # For fetch price --> get the full list and then process in order to reduce the number of API calls
    def fetch_ticker_hist_price(symbol):
        endpoint = "historical-price-eod/full"
        query = f"symbol={symbol}"
        df_hist_price = API.fetch(endpoint,query)
        return API.set_index_and_sort(df_hist_price)
    
    def set_index_and_sort(df,index="date"):
        df.set_index('date', inplace=True)
        df.index = pd.to_datetime(df.index)
        df = df.sort_index(ascending=True)
        return df

In [ ]:
class Dir:

    ROOT = Path.cwd() / "data"
    RAW = ROOT / 'raw'
    INTERIM = ROOT /  'interim'
    PROCESSED = ROOT / 'processed'
    EXTERNAL = ROOT / 'external'

# file names
class CSV:
    SP500 = 'sp500_constituent.csv'
    KEY_METRIC = 'key_metrics.csv'
    RATIOS = 'ratios.csv'
    STOCK_HIST = 'stock_hist.csv'
    GROWTH = 'growth.csv' # basically merge of key_metrics and ratios csv

class FolderStructure:

    @staticmethod
    def create():
        Dir.ROOT.mkdir(parents=True, exist_ok=True)
        #FolderStructure.download_sp500_constituent_info() # need to be added later
        Dir.RAW.mkdir(parents=True, exist_ok=True)
        Dir.EXTERNAL.mkdir(parents=True, exist_ok=True)
        Dir.PROCESSED.mkdir(parents=True, exist_ok=True)
        Dir.INTERIM.mkdir(parents=True, exist_ok=True)

    @staticmethod
    def load_data():
        df = FolderStructure.read_external_csv(CSV.SP500)
        print("Loading data...")
        for index, row in df.iterrows():
            print(f"\t[{index}/500] - {row['symbol']}" )
            FolderStructure.download_ticker_raw_data(row['symbol'], 'anual', add=row['sector'])
            break


    @staticmethod
    def read_ticker_csv(stage_dir, symbol, csv_name, index='date'):
        target_dir = stage_dir / 'tickers' / symbol
        target_dir.mkdir(parents=True, exist_ok=True)
        df = pd.read_csv(target_dir / csv_name, index_col=index)
        return df

    @staticmethod
    def write_ticker_csv(df, stage_dir, symbol, csv_name):
        target_dir = stage_dir / 'tickers' / symbol
        target_dir.mkdir(parents=True, exist_ok=True)
        df.to_csv(target_dir / csv_name)

    @staticmethod
    def read_raw_ticker_csv(symbol, csv_name):
        return FolderStructure.read_ticker_csv(Dir.RAW, symbol, csv_name)

    @staticmethod
    def read_interim_ticker_csv(symbol, csv_name, index='date'):
        target_dir = Dir.INTERIM / 'tickers'
        df = pd.read_csv(target_dir / csv_name, index_col=index) # WARNING might cause issues --> rows with same indexes
        return df
    
    @staticmethod
    def read_external_csv(csv_name):
        target_dir = Dir.EXTERNAL
        target_dir.mkdir(parents=True, exist_ok=True)
        df = pd.read_csv(target_dir / csv_name)
        return df


    @staticmethod
    def write_processed_ticker_csv(df, symbol, csv_name):
        FolderStructure.write_ticker_csv(df, Dir.PROCESSED, symbol, csv_name)

    @staticmethod
    def download_sp500_constituent_info():
        df = API.fetch_sp500_constituent_info()
        df.to_csv(Dir.EXTERNAL / CSV.SP500)

    @staticmethod
    def download_ticker_raw_data(symbol, period="quarter", add=""):
        parent = Dir.RAW / 'tickers' / symbol
        parent.mkdir(parents=True, exist_ok=True)
        df_stock_hist = API.fetch_ticker_hist_price(symbol)
        df_stock_hist.to_csv(parent / CSV.STOCK_HIST)

        df_key_metrics = API.fetch_ticker_key_metrics(symbol, period)
        df_key_metrics.to_csv(parent / CSV.KEY_METRIC)

        df_ratios = API.fetch_ticker_financial_ratios(symbol, period)
        df_ratios.to_csv(parent / CSV.RATIOS)

    @staticmethod
    def process_ticker_data(symbol, period="quarter"):
        raw_dir = Dir.RAW / 'tickers' / symbol
        raw_dir.mkdir(parents=True, exist_ok=True)

        train_dir = Dir.PROCESSED / 'train'
        train_dir.mkdir(parents=True, exist_ok=True)

        test_dir = Dir.PROCESSED / 'test'
        test_dir.mkdir(parents=True, exist_ok=True)


In [193]:
FolderStructure.create()
FolderStructure.load_data()

Loading data...
	[0/500] - AAPL


In [ ]:
class Signals:

    EXCLUDE_COLS = ['symbol','fiscalYear','period','reportedCurrency'] 


    def construct_features(symbol,period='quarter'):
        df_label = pd.DataFrame({'y':[]})
        df_all_info = Signals.growth_statements_info(symbol,period)
        df_stock = Signals.stock_insight(symbol)
        df_stock.index = pd.to_datetime(df_stock.index)

        df_all_info = df_all_info.merge(df_stock,on='date')
        df_label["y"] = df_all_info['ret_30d']
        print(df_label)
        df_label = df_label.shift(-1)
        df_all_info['y'] = df_label

        FolderStructure.write_ticker_csv(df_all_info,Dir.INTERIM,symbol,f"{symbol}.csv")

    def growth_statements_info(symbol,period='quarter',num_periods_compared=1):

        df_growth_info_raw = FolderStructure.read_raw_ticker_csv(symbol,CSV.RATIOS)
        df_metrics_raw = FolderStructure.read_raw_ticker_csv(symbol,CSV.KEY_METRIC)

        #df_growth_info_raw = API.fetch_ticker_financial_ratios(symbol,period)
        #df_metrics_raw = API.fetch_ticker_key_metrics(symbol,period) 
        df_growth_info_raw.merge(df_metrics_raw,on='date')
        df_growth_info = df_growth_info_raw[Signals.EXCLUDE_COLS]

        for gap in range(1,num_periods_compared+1):
            df_p_growth = df_growth_info_raw.drop(columns=Signals.EXCLUDE_COLS).pct_change(gap)
            new_columns_names = [name+"_"+str(gap)+"p" for name in df_p_growth.columns]
            df_p_growth.columns = new_columns_names
            df_growth_info.loc[:,df_p_growth.columns] = df_p_growth
        
        # replace end of period dates by business days since sometimes they are on weekends
        df_growth_info = df_growth_info.reset_index()
        df_growth_info['date'] = pd.to_datetime(df_growth_info['date'])
        df_growth_info['date']= df_growth_info['date'].apply(lambda d: d if d.weekday() < 5 else d - pd.offsets.BDay(1))

        df_growth_info.set_index('date',inplace=True)
  
        #FolderStructure.write_processed_ticker_csv(df_growth_info,symbol,CSV.GROWTH) TESTING
        return df_growth_info
    
    def stock_insight(symbol): # Determine Analysis Period
        #df_hist_raw = API.fetch_ticker_hist_price(symbol)
        df_hist_raw = FolderStructure.read_raw_ticker_csv(symbol,CSV.STOCK_HIST)

        # For dataframe subtractions collumns must have same names. This is not the case for series
        closing_price = df_hist_raw['close']
        low_price = df_hist_raw['low']
        high_price = df_hist_raw['high']
        volume = df_hist_raw["volume"]

        # Return
        ret_1d = closing_price.pct_change()
        ret_2d = closing_price.pct_change(2)
        ret_5d = closing_price.pct_change(5)
        ret_10d = closing_price.pct_change(10)
        ret_20d = closing_price.pct_change(20)
        ret_30d = closing_price.pct_change(30)

        # Volatility
        vol_5d = closing_price.pct_change().shift(1).rolling(window=5).std()
        vol_10d = closing_price.pct_change().shift(1).rolling(window=10).std()

        # Momentum
        momentum_10d = closing_price - closing_price.shift(10)

        # MFV
        mfm = ((closing_price-low_price) - (high_price-closing_price))/(high_price-low_price)
        mfv = mfm * volume
        ad_line = mfv.cumsum()
        AD_momentum_5d = ad_line.pct_change(5)
        AD_momentum_10d = ad_line.pct_change(10)
        AD_momentum_20d = ad_line.pct_change(20)

        # SMA Ratio
        sma_10 = closing_price.shift(1).rolling(window=10).mean()
        sma_50 = closing_price.shift(1).rolling(window=50).mean()
        sma_ratio = sma_10/sma_50

        # Z-score (20d)
        rolling_mean = closing_price.shift(1).rolling(window=20).mean()
        rolling_std = closing_price.shift(1).rolling(window=20).std()
        z_score_20d = (closing_price - rolling_mean)/rolling_std

        # RSI (14d)
        delta = closing_price.diff()
        gain = delta.where(delta > 0, 0.0)
        loss = -delta.where(delta < 0, 0.0)

        avg_gain = gain.shift(1).rolling(window=14).mean()
        avg_loss = loss.shift(1).rolling(window=14).mean()

        rs = avg_gain/avg_loss
        rsi_14 = 100 - (100 / (1 + rs))
        
        dataframes = [
        (ret_1d, 'ret_1d'),
        (ret_2d, 'ret_2d'),
        (ret_5d, 'ret_5d'),
        (ret_10d, 'ret_10d'),
        (ret_20d, 'ret_20d'),
        (ret_30d, 'ret_30d'),
        (vol_5d, 'vol_5d'),
        (vol_10d, 'vol_10d'),
        (momentum_10d, 'momentum_10d'),
        (sma_ratio, 'sma_ratio_10_50'),
        (z_score_20d, 'zscore_20d'),
        (rsi_14, 'rsi_14'),
        (volume,'volume'),
        (AD_momentum_5d,'AD_momentum_5d'),
        (AD_momentum_10d,'AD_momentum_10d'),
        (AD_momentum_20d,'AD_momentum_20d')
        ]


        for column, name in dataframes:
            column.name = name

        df_stock_insight = pd.concat(
            [column for column, _ in dataframes],
            axis=1
        )
        
        #FolderStructure.write_processed_ticker_csv(df_stock_insight,symbol,CSV.STOCK_HIST) # TESTING
        return df_stock_insight

    def process_data():
        df = FolderStructure.read_external_csv(CSV.SP500)
        print("Processing data...")
        for index, row in df.iterrows():
            print(f"\t[{index}/500] - {row['symbol']}" )
            Signals.construct_features(row['symbol'],period='anual')
            break # REMOVE
    
    # NEED TO CONSIDER HOW TO SEPERATE THE DATA

In [197]:
Signals.process_data()


Processing data...
	[0/500] - AAPL
                   y
date                
2020-09-25 -0.023737
2021-09-24 -0.013231
2022-09-23 -0.107187
2023-09-29 -0.016034
2024-09-27  0.013661


# Feature Construction
- Need to perform this for Balance Sheet, Income Statement and Historical Stock Price
- Need a way to handle a variety of stocks (ticker feature)

In [154]:
# Feature Construction
from sklearn.base import BaseEstimator, TransformerMixin

# Need to perform this for Balance Sheet, Income Statement and Historical Stock Price
class TechnicalIndicatorTransformer(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass

    def fit(self, X, y=None):
        return self

    def transform(self, X): # X are the fundamental values needed to calculate all the other features
        df = X.copy()
        
        # Create many features inside one transformer
        df['sma_10'] = df['close'].rolling(10).mean()
        df['sma_50'] = df['close'].rolling(50).mean()
        df['rsi'] = compute_rsi(df['close'])
        df['macd'] = compute_macd(df['close'])
        df['lag_1'] = df['close'].shift(1)
        df['volatility'] = df['close'].rolling(10).std()
        # ... up to 140 features

        return df # handle NaNs from rolling/shift

# Data Cleaning + PreProcessing
- NaN values will be dropped

In [278]:
from sklearn.pipeline import FeatureUnion, Pipeline

transformer1 = Pipeline([
    ('imputer', SimpleImputer()),
    ('scaler', StandardScaler())
])

transformer2 = Pipeline([
    ('pca', PCA(n_components=5))
])


NameError: name 'SimpleImputer' is not defined

# Feature Ranking